In [1]:

#Import Libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")

pd.set_option("display.max_columns", None)

In [7]:
#Load Clean Dataset

df = pd.read_csv("fct_abduction_cleaned.csv")

df["DATE"] = pd.to_datetime(df["DATE"])

df.head()

,DATE,THREAT ACTORS,AFFECTED GROUPS,SECURITY INCIDENT TYPE,SECURITY INCIDENT SUB-TYPE,STATE,LAC,LOCATION,NUMBER OF PEOPLE KIDNAPPED,NUMBER OF PEOPLE KILLED,NUMBER OF PEOPLE INJURED,BREAKDOWN OF INCIDENT,MONTH,YEAR,DAY,DAY_OF_WEEK
0,2021-01-23,Criminals,Civilians,Abduction,Kidnapping,Federal Capital Territory,Abaji,Other,8.0,0.0,0.0,"Gunmen attack orphanage home, abduct 10 in Abuja",1,2021,23,Saturday
1,2021-02-02,Criminals,Civilians,Abduction,Kidnapping,Federal Capital Territory,Bwari,Residence,3.0,0.0,0.0,"Gunmen kidnap father of Council chair, 2 sibli...",2,2021,2,Tuesday
2,2021-02-26,Criminals,Civilians,Abduction,Kidnapping,Federal Capital Territory,Bwari,Residence,2.0,0.0,0.0,"Anxiety as gunmen invade Abuja community, abdu...",2,2021,26,Friday
3,2021-03-14,Criminals,Civilians,Abduction,Kidnapping,Federal Capital Territory,Abaji,Community,4.0,0.0,0.0,Two killed as police foil kidnap attempt in Ab...,3,2021,14,Sunday
4,2021-03-24,Criminals,Civilians,Abduction,Kidnapping,Federal Capital Territory,Kuje,Residence,4.0,0.0,0.0,"Gunmen abduct 4 persons in Abuja, demand N200m...",3,2021,24,Wednesday


In [9]:
#Executive Summary
total_incidents = len(df)

total_kidnapped = df["NUMBER OF PEOPLE KIDNAPPED"].sum()

total_killed = df["NUMBER OF PEOPLE KILLED"].sum()

total_injured = df["NUMBER OF PEOPLE INJURED"].sum()

affected_states = df["STATE"].nunique()

print("="*50)
print("EXECUTIVE SUMMARY")
print("="*50)

print(f"Total Incidents: {total_incidents:,}")

print(f"People Kidnapped: {int(total_kidnapped):,}")

print(f"People Killed: {int(total_killed):,}")

print(f"People Injured: {int(total_injured):,}")

print(f"States Covered: {affected_states}")

EXECUTIVE SUMMARY
Total Incidents: 164
People Kidnapped: 718
People Killed: 34
People Injured: 19
States Covered: 1


In [11]:
#Create a Risk Score

df["RISK_SCORE"] = (
    df["NUMBER OF PEOPLE KIDNAPPED"]
    + (df["NUMBER OF PEOPLE KILLED"] * 3)
    + (df["NUMBER OF PEOPLE INJURED"] * 2)
)

In [19]:
#Highest Risk States
risk_state = (
    df.groupby("STATE")
      .agg(
          Incidents=("STATE","count"),
          Kidnapped=("NUMBER OF PEOPLE KIDNAPPED","sum"),
          Killed=("NUMBER OF PEOPLE KILLED","sum"),
          Injured=("NUMBER OF PEOPLE INJURED","sum"),
          Risk=("RISK_SCORE","sum")
      )
      .sort_values("Risk",ascending=False)
)

risk_state

,Incidents,Kidnapped,Killed,Injured,Risk
STATE,,,,,
Federal Capital Territory,164,718.0,34.0,19.0,858.0


In [21]:
#Highest Risk LGAs

risk_lga = (
    df.groupby(["STATE","LAC"])
      .agg(
          Incidents=("LAC","count"),
          Kidnapped=("NUMBER OF PEOPLE KIDNAPPED","sum"),
          Killed=("NUMBER OF PEOPLE KILLED","sum"),
          Risk=("RISK_SCORE","sum")
      )
      .sort_values("Risk",ascending=False)
)

risk_lga.head(15)

Incidents  Kidnapped  Killed   Risk
STATE                     LAC                                                 
Federal Capital Territory Bwari                   49      265.0    14.0  331.0
                          Abuja Municipal         53      166.0     4.0  184.0
                          Kuje                    20      113.0     8.0  139.0
                          Abaji                   17       77.0     6.0  101.0
                          Kwali                   18       78.0     1.0   81.0
                          Gwagwalada               6       18.0     1.0   21.0
                          Karu                     1        1.0     0.0    1.0

In [23]:
#Most Active Threat Actors

actors = (
    df.groupby("THREAT ACTORS")
      .agg(
          Incidents=("THREAT ACTORS","count"),
          Kidnapped=("NUMBER OF PEOPLE KIDNAPPED","sum"),
          Killed=("NUMBER OF PEOPLE KILLED","sum"),
          Risk=("RISK_SCORE","sum")
      )
      .sort_values("Risk",ascending=False)
)

actors

,Incidents,Kidnapped,Killed,Risk
THREAT ACTORS,,,,
Criminals,106,443.0,19.0,528.0
NSAG,39,208.0,5.0,223.0
Bandits,11,57.0,7.0,86.0
Unknown,2,3.0,2.0,9.0
Political Thug,1,1.0,1.0,6.0
Civilians,3,4.0,0.0,4.0
Humanitarian Aid Worker,1,1.0,0.0,1.0
Security Forces,1,1.0,0.0,1.0


In [25]:
#Monthly Trend

monthly = (
    df.groupby(["YEAR","MONTH"])
      .size()
      .reset_index(name="Incidents")
)

monthly

,YEAR,MONTH,Incidents
0,2021,1,1
1,2021,2,2
2,2021,3,3
3,2021,4,4
4,2021,5,2
5,2021,6,2
6,2021,7,1
7,2021,8,1
8,2021,9,3
9,2021,10,1


In [27]:
#Weekday Analysis

weekday = (
    df.groupby("DAY_OF_WEEK")
      .size()
      .sort_values(ascending=False)
)

weekday

DAY_OF_WEEK
Tuesday      31
Sunday       30
Wednesday    29
Monday       21
Saturday     21
Thursday     21
Friday       11
dtype: int64

In [29]:
#Victim Profile

victims = (
    df.groupby("AFFECTED GROUPS")
      .agg(
          Incidents=("AFFECTED GROUPS","count"),
          Kidnapped=("NUMBER OF PEOPLE KIDNAPPED","sum")
      )
      .sort_values("Kidnapped",ascending=False)
)

victims

,Incidents,Kidnapped
AFFECTED GROUPS,,
Civilians,158,676.0
Farmer/Herder,2,25.0
"Security Forces (Military, Police, Paramilitary, DSS / SSS and including MNJTF)",3,16.0
Government Official,1,1.0


In [31]:
#Operational Recommendations

highest_state = risk_state.index[0]

highest_actor = actors.index[0]

highest_day = weekday.index[0]

print("="*60)
print("STRATEGIC RECOMMENDATIONS")
print("="*60)

print(
    f"1. Prioritize security operations in {highest_state}, "
    "which recorded the highest overall risk score."
)

print(
    f"2. Increase intelligence gathering against {highest_actor}, "
    "identified as the most impactful threat actor."
)

print(
    f"3. Strengthen patrols during periods corresponding to "
    f"{highest_day}, when incidents were most frequent."
)

print(
    "4. Allocate emergency response resources based on risk "
    "scores rather than incident counts alone."
)

print(
    "5. Continue monitoring monthly trends to detect emerging "
    "hotspots before they escalate."
)

STRATEGIC RECOMMENDATIONS
1. Prioritize security operations in Federal Capital Territory, which recorded the highest overall risk score.
2. Increase intelligence gathering against Criminals, identified as the most impactful threat actor.
3. Strengthen patrols during periods corresponding to Tuesday, when incidents were most frequent.
4. Allocate emergency response resources based on risk scores rather than incident counts alone.
5. Continue monitoring monthly trends to detect emerging hotspots before they escalate.


In [33]:
#Exporting these summary tables:
risk_state.to_csv("state_risk_summary.csv")

risk_lga.to_csv("lga_risk_summary.csv")

actors.to_csv("threat_actor_summary.csv")

monthly.to_csv("monthly_trend.csv", index=False)

victims.to_csv("victim_profile.csv")